# 03 — `@dataclass` and `@property`

Two decorators that remove huge piles of OOP boilerplate. `@dataclass` writes
`__init__`/`__repr__`/`__eq__` for you. `@property` lets you expose a *computed* value as if it were an attribute.

These are everywhere in modern Python — Pydantic models build on top of `@dataclass`, FastAPI dependency results are often dataclasses, agent tool outputs are usually dataclasses or Pydantic models.

## The boilerplate `@dataclass` replaces

In [1]:
# Without dataclass — typical hand-written value object:
class Point:
    def __init__(self, x: float, y: float):
        self.x = x
        self.y = y
    def __repr__(self):
        return f"Point(x={self.x!r}, y={self.y!r})"
    def __eq__(self, other):
        return isinstance(other, Point) and (self.x, self.y) == (other.x, other.y)
    def __hash__(self):
        return hash((self.x, self.y))

p1 = Point(3, 4)
p2 = Point(3, 4)
print(p1, p1 == p2)

Point(x=3, y=4) True


In [2]:
# Same thing with @dataclass — all four dunders generated:
from dataclasses import dataclass

@dataclass(frozen=True)              # frozen=True also generates __hash__
class Point:
    x: float
    y: float

p1 = Point(3, 4)
p2 = Point(3, 4)
print(p1, p1 == p2)
print({p1, p2})                      # hashable — duplicates collapse

Point(x=3, y=4) True
{Point(x=3, y=4)}


## `@dataclass` flags worth knowing

In [3]:
from dataclasses import dataclass, field

# kw_only=True forces named arguments. frozen=True forbids mutation after __init__.
@dataclass(frozen=True, kw_only=True)
class User:
    name: str
    email: str
    is_admin: bool = False
    # default_factory generates a FRESH default value per instance — no mutable-default trap.
    tags: list[str] = field(default_factory=list)

u = User(name="Alice", email="a@b.c", tags=["early"])
print(u)

# kw_only=True means you must pass everything by keyword — readable and safe.
try:
    User("Alice", "a@b.c")
except TypeError as e:
    print("kw_only rejected positional:", e)

# frozen=True means you cannot reassign fields after construction.
try:
    u.email = "x@y.z"
except Exception as e:    # FrozenInstanceError is a subclass of AttributeError
    print("frozen rejected reassignment:", type(e).__name__, "-", e)

User(name='Alice', email='a@b.c', is_admin=False, tags=['early'])
kw_only rejected positional: User.__init__() takes 1 positional argument but 3 were given
frozen rejected reassignment: FrozenInstanceError - cannot assign to field 'email'


### `slots=True` — no surprise attributes

A separate cell because combining `slots=True` with `frozen=True` hits a known Python 3.12 quirk in some cases. On its own, `slots=True` makes typos fail loudly — no more silent `u.emial = "..."` writing a junk attribute that you discover an hour later.

In [4]:
from dataclasses import dataclass

@dataclass(slots=True)
class Config:
    host: str
    port: int

cfg = Config("localhost", 8080)
print(cfg)

# Reassigning a declared field is fine (no frozen=True here):
cfg.port = 9000
print(cfg)

# But adding an UNDECLARED attribute is rejected — typos surface immediately.
try:
    cfg.hsot = "oops"     # typo'd host
except AttributeError as e:
    print("slots rejected new attr:", e)

Config(host='localhost', port=8080)
Config(host='localhost', port=9000)
slots rejected new attr: 'Config' object has no attribute 'hsot'


### `field(default_factory=...)` — the safe way to default a list/dict

Same idea as the mutable-default trap from `02_functions_deep`. **Never** write `tags: list[str] = []` on a dataclass — it's actually rejected by `@dataclass` since 3.11+ to prevent the bug.

In [5]:
from dataclasses import dataclass, field

@dataclass
class Bag:
    items: list[str] = field(default_factory=list)    # fresh list every Bag()

a, b = Bag(), Bag()
a.items.append("x")
print(a.items, b.items)     # ['x'] []

['x'] []


## `@dataclass` plays well with methods

It's not just data. You can still add methods — `@dataclass` only generates the dunders, it doesn't restrict the rest of the class.

In [6]:
import math
from dataclasses import dataclass

@dataclass(frozen=True)
class Vector:
    x: float
    y: float

    def __add__(self, other: "Vector") -> "Vector":
        return Vector(self.x + other.x, self.y + other.y)

    def __sub__(self, other: "Vector") -> "Vector":
        return Vector(self.x - other.x, self.y - other.y)

    def magnitude(self) -> float:
        return math.hypot(self.x, self.y)

a = Vector(3, 4)
b = Vector(1, 1)
print(a + b)
print(a - b)
print(a.magnitude())

Vector(x=4, y=5)
Vector(x=2, y=3)
5.0


## `@property` — a computed attribute

Sometimes you want callers to write `obj.foo` (no parens) but you need to *compute* `foo`. `@property` turns a method into a read-only attribute.

In [7]:
import math
from dataclasses import dataclass

@dataclass(frozen=True)
class Circle:
    radius: float

    @property
    def area(self) -> float:
        return math.pi * self.radius ** 2

    @property
    def diameter(self) -> float:
        return self.radius * 2

c = Circle(3)
print(c.area)         # no parens — looks like an attribute
print(c.diameter)

# Read-only by default — assigning to it raises:
try:
    c.area = 0
except (AttributeError, Exception) as e:
    print("rejected:", e)

28.274333882308138
6
rejected: cannot assign to field 'area'


### `@property` with a setter — validated writes

Adding a `.setter` lets the property be assignable, with validation in one place. Compare this to writing `get_temperature()` / `set_temperature(value)` methods.

In [8]:
class Sensor:
    ABSOLUTE_ZERO_C = -273.15

    def __init__(self, celsius: float):
        self.temperature = celsius   # triggers the SETTER below

    @property
    def temperature(self) -> float:
        return self._temp

    @temperature.setter
    def temperature(self, value: float) -> None:
        if value < self.ABSOLUTE_ZERO_C:
            raise ValueError(f"below absolute zero: {value}")
        self._temp = value     # the actual storage; convention: leading underscore = "internal"

s = Sensor(20.0)
print(s.temperature)
s.temperature = -50          # valid
print(s.temperature)
try:
    s.temperature = -300     # below absolute zero
except ValueError as e:
    print("rejected:", e)

20.0
-50
rejected: below absolute zero: -300


## Mini-exercise

1. Build `@dataclass(frozen=True) class Money(amount: float, currency: str)`. Add `__add__` that only works between same currencies (raise `ValueError` otherwise). Demonstrate that two equal `Money` values are equal and hashable.
2. Add a `@property` `is_zero` to `Money` returning `bool`.
3. Convert the `BankAccount` from [03_oop_basics/03_bank_account/account.py](../03_oop_basics/03_bank_account/account.py) to a `@dataclass`. What stays, what changes? Why might you *not* use `@dataclass` for it? (Hint: `__init__` does non-trivial validation; dataclasses' generated `__init__` doesn't.)
4. Predict, then run:
   ```python
   from dataclasses import dataclass
   @dataclass
   class Bag:
       items: list[str] = []     # what happens?
   ```

In [9]:
# your solutions here


**Takeaways**

- `@dataclass` generates `__init__` / `__repr__` / `__eq__` from your typed attributes.
- `frozen=True` → immutable + hashable. `slots=True` → no surprise attrs. `kw_only=True` → forces named args.
- Use `field(default_factory=list)` for mutable defaults. **Never** `= []`.
- `@property` exposes a derived value as an attribute (`obj.area`, no parens).
- `@x.setter` lets you validate writes in one place.

Next: [04_shapes_app/](04_shapes_app/) — put it all together in a real .py app.